## Verify Tensorflow

This notebook verifies that Tensorflow is working on the GPU.

## Imports

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

2025-11-25 17:21:31.504824: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Check GPU driver and Cuda Toolkit

In [2]:
!nvidia-smi
!nvcc -V

Tue Nov 25 17:21:42 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.274.02             Driver Version: 535.274.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        Off | 00000000:41:00.0 Off |                  N/A |
|  0%   28C    P8              17W / 420W |    334MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## Verify GPU and CUDA access from Tensorflow

In [3]:
# Basic version + GPU info
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs detected:", gpus)

print()

# Try to get CUDA / cuDNN versions (if available in this build)
try:
    from tensorflow.python.platform import build_info as tf_build_info
    print("CUDA version:", getattr(tf_build_info, "cuda_version_number", "Unknown"))
    print("cuDNN version:", getattr(tf_build_info, "cudnn_version_number", "Unknown"))
except Exception as e:
    print("Could not query CUDA/cuDNN versions from TensorFlow build info:", e)

print()

# Simple device info + matmul test
if gpus:
    print(f"Number of GPUs: {len(gpus)}")
    print("Using device: GPU:0")
    device_name = "/GPU:0"
else:
    print("No GPU detected by TensorFlow, using CPU.")
    device_name = "/CPU:0"

print()

with tf.device(device_name):
    x = tf.random.normal((3, 3))
    y = tf.random.normal((3, 3))
    z = tf.matmul(x, y)

print()

print("x:", x)
print()
print("y:", y)
print()
print("x @ y:", z)

TensorFlow version: 2.20.0
GPUs detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

CUDA version: Unknown
cuDNN version: Unknown

Number of GPUs: 1
Using device: GPU:0


x: tf.Tensor(
[[-0.741118    1.3840798   0.26720214]
 [ 1.1205012   1.0179523   0.5957313 ]
 [-0.94256926 -0.75018936  0.42485744]], shape=(3, 3), dtype=float32)

y: tf.Tensor(
[[-0.49103156 -0.07300895 -0.04600435]
 [ 0.07756827 -0.72839934 -0.19017553]
 [ 0.36679476 -0.7396301   0.79102165]], shape=(3, 3), dtype=float32)

x @ y: tf.Tensor(
[[ 0.56928134 -1.1516854  -0.01776077]
 [-0.25272948 -1.2639033   0.22609882]
 [ 0.5604758   0.3010161   0.5221014 ]], shape=(3, 3), dtype=float32)


I0000 00:00:1764091349.155756    7800 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22132 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:41:00.0, compute capability: 8.6
